In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Section 1: Loading data and validation

In [8]:
path = '/Users/kaungkhantkyaw/Documents/GitHub/Analyse-US-Domestic-flights/Data/Cleaned/DB1B_Market_model_ready.parquet'
df_model = pd.read_parquet(path)

print(df_model.shape)
display(df_model.head())
display(df_model.sample(5, random_state = 42))
display(df_model.info())

(7189747, 15)


,Origin,Dest,OriginState,DestState,TkCarrier,Passengers,MktDistance,MktCoupons,MktDistanceGroup,MktGeoType,extra_miles,carrier_mismatch,log_passengers,distance_per_coupon,MktFare
0,SAN,ATL,CA,GA,DL,1.0,1892.0,1,4,2,0.0,0,0.693147,1892.0,351.5
1,ATL,SAN,GA,CA,DL,2.0,1892.0,1,4,2,0.0,0,1.098612,1892.0,352.0
2,SAN,ATL,CA,GA,DL,2.0,1892.0,1,4,2,0.0,0,1.098612,1892.0,352.0
3,ATL,SAN,GA,CA,DL,1.0,1892.0,1,4,2,0.0,0,0.693147,1892.0,352.5
4,SAN,ATL,CA,GA,DL,1.0,1892.0,1,4,2,0.0,0,0.693147,1892.0,352.5


,Origin,Dest,OriginState,DestState,TkCarrier,Passengers,MktDistance,MktCoupons,MktDistanceGroup,MktGeoType,extra_miles,carrier_mismatch,log_passengers,distance_per_coupon,MktFare
1799499,GRR,AUS,MI,TX,DL,1.0,1269.0,2,3,2,166.0,1,0.693147,634.5,5.13
7057535,HSV,RDU,AL,NC,DL,1.0,507.0,2,2,2,47.0,0,0.693147,253.5,307.00
4843690,ORD,DEN,IL,CO,WN,1.0,888.0,1,2,2,0.0,0,0.693147,888.0,210.00
6002858,MCO,GJT,FL,CO,UA,1.0,1758.0,2,4,2,36.0,0,0.693147,879.0,522.00
878153,BIL,BFL,MT,CA,AA,1.0,1298.0,2,3,2,395.0,1,0.693147,649.0,194.00


<class 'pandas.DataFrame'>
RangeIndex: 7189747 entries, 0 to 7189746
Data columns (total 15 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Origin               str    
 1   Dest                 str    
 2   OriginState          str    
 3   DestState            str    
 4   TkCarrier            str    
 5   Passengers           float64
 6   MktDistance          float64
 7   MktCoupons           int64  
 8   MktDistanceGroup     int64  
 9   MktGeoType           int64  
 10  extra_miles          float64
 11  carrier_mismatch     int64  
 12  log_passengers       float64
 13  distance_per_coupon  float64
 14  MktFare              float64
dtypes: float64(6), int64(4), str(5)
memory usage: 905.1 MB


None

Section 2: Sample & Separate

Theres almost 7.2 million rows, thus sampling is needed to prevent long training times and using alot of memory.
The sample dataset is also compared to the full set to ensure consistentcy in values.


In [11]:
sample_size = 100_000
target = 'MktFare'

if len(df_model) > sample_size:
    df_sample = df_model.sample(sample_size, random_state = 42).copy()
else:
    df_sample = df_model.copy()

In [12]:
sample_comparison = pd.DataFrame({
    'full_dataset': df_model['MktFare'].describe(),
    'sampled_dataset': df_sample['MktFare'].describe()
})

display(sample_comparison)

quantiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]

sample_comparison_quantiles = pd.DataFrame({
    'full_dataset': df_model['MktFare'].quantile(quantiles),
    'sampled_dataset': df_sample['MktFare'].quantile(quantiles)
})

display(sample_comparison_quantiles)


,full_dataset,sampled_dataset
count,7.189747e+06,100000.000000
mean,2.641319e+02,264.463935
std,1.741388e+02,174.320416
min,5.000000e+00,5.000000
25%,1.500000e+02,150.710000
50%,2.331700e+02,233.285000
75%,3.435000e+02,343.500000
max,1.142000e+03,1142.000000


,full_dataset,sampled_dataset
0.01,5.000,5.000
0.05,25.313,25.000
0.25,150.000,150.710
0.50,233.170,233.285
0.75,343.500,343.500
0.95,591.500,593.000
0.99,883.500,884.000


In [14]:
x = df_sample.drop(columns = [target]).copy()
y = df_sample[target].copy()

print("Is target included in x: ", target in x.columns)
print("Features shape: ", x.shape)
print("Target shape: ", y.shape)
print("Target median: ", y.median())

Is target included in x:  False
Features shape:  (100000, 14)
Target shape:  (100000,)
Target median:  233.285


Section 3: Train/Test split

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)
#x_train is used to train the model, y_train provides the fare values corresponding to it. x_test is the
#unseen values used to test the model, y_test provides the actual fares used to check test predictions.
#80% training data, 20% test data split

print('x_train shape: ', x_train.shape)
print('x_test shape: ', x_test.shape)
print('y_train shape: ', y_train.shape)
print('y_test shape: ', y_test.shape)


x_train shape:  (80000, 14)
x_test shape:  (20000, 14)
y_train shape:  (80000,)
y_test shape:  (20000,)


In [ ]:
cat_cols = ['Origin', 'Dest', 'OriginState', 'DestState', 'TkCarrier', 
'MktDistanceGroup', 'MktGeoType', 'carrier_mismatch']

num_cols = ['Passengers', 'MktDistance', 'MktCoupons', 'extra_miles', 'log_passengers', 'distance_per_coupon']

selected_cols = cat_cols + num_cols

print('Number of selected features: ', len(selected_cols))
print('Number of x columns: ', x.shape[1])

Number of selected features:  14
Number of x columns:  14


In [21]:
cat_transformer = OneHotEncoder(handle_unknown = 'ignore') #handle_unknown = 'ignore' is used to ignore situations where a rare unseen category appears in test, letting the code run
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(transformers = 
[('categorical', cat_transformer, cat_cols), ('numeric', num_transformer, num_cols)], remainder = 'drop')
display(preprocessor)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_na

Section 4: Baseline and Regression model

In [23]:
dummy_model = DummyRegressor(strategy = 'median') #the model predicts the same median fare for every row, median used as more robust
dummy_model.fit(x_train, y_train)
dummy_pred = dummy_model.predict(x_test)

display(dummy_pred[:10])


array([233.01, 233.01, 233.01, 233.01, 233.01, 233.01, 233.01, 233.01,
       233.01, 233.01])

In [24]:
dummy_mae = mean_absolute_error(y_test, dummy_pred)
#average abs difference between predicted and actual fares
dummy_rmse = mean_squared_error(y_test, dummy_pred)**0.5
#Provides large errors more influences, making detecting them easier
dummy_r2 = r2_score(y_test, dummy_pred)


In [25]:
def evaluate_regression_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)**0.5
    r2 = r2_score(y_true, y_pred)

    return{'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

In [26]:
dummy_results = evaluate_regression_model(model_name = 'Dummy Median', y_true = y_test, y_pred = dummy_pred)
display(dummy_results)

model_results = []
model_results.append(dummy_results)


{'Model': 'Dummy Median',
 'MAE': 126.096694,
 'RMSE': 176.96874842553416,
 'R2': -0.03271356183254248}

In [27]:
linear_pipeline = Pipeline(steps = [('preprocessor', preprocessor), ('model', LinearRegression())])

linear_pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [28]:
linear_pred = linear_pipeline.predict(x_test)
display(linear_pred.shape)
display(linear_pred[:10])

(20000,)

array([308.14781557, 348.14418393, 115.87278461, 174.92030863,
       264.26363501, 200.28241685, 222.96357475, 270.47391141,
       264.0352995 , 299.13004647])

In [ ]:
display((linear_pred < 0).sum())
#negative predictions are not realistic

np.int64(65)

In [ ]:
linear_results = evaluate_regression_model(model_name = 'Linear Regression', y_true = y_test, y_pred = linear_pred)
display(linear_results)

model_results.append(linear_results)
results_df = pd.DataFrame(model_results)
display(results_df)

mae_improvement = ((dummy_results['MAE'] - linear_results['MAE'] / dummy_results['MAE']) * 100)
print(f"MAE improvement over baseline/dummy model: {mae_improvement:.2f}%")

{'Model': 'Linear Regression',
 'MAE': 111.6290228941815,
 'RMSE': 155.3474866709794,
 'R2': 0.20421603473457217}

,Model,MAE,RMSE,R2
0,Dummy Median,126.096694,176.968748,-0.032714
1,Linear Regression,111.629023,155.347487,0.204216


Linear regression performed better than the dummy baseline, reducing MAE and RMSE. The R2 value of approximately 0.20 shows that it only explains 20% of the market fare varaitions, which concludes that the relationship between features and fares are more complex and non-linear

In [32]:
#initial residual analysis by comapring actual and predicted fares
prediction_comparison = pd.DataFrame({
    'ActualFare': y_test.to_numpy(),
    'PredictedFare': linear_pred
})
display(prediction_comparison.head(10))

prediction_comparison['Residual'] = (prediction_comparison['ActualFare'] - prediction_comparison['PredictedFare'])
prediction_comparison['AbsoluteResidual'] = prediction_comparison['Residual'].abs()
display(prediction_comparison.head(10))
#positive residual means underpredict, negative residual means overpredict, near 0 means accurate

,ActualFare,PredictedFare
0,300.50,308.147816
1,143.50,348.144184
2,67.50,115.872785
3,138.00,174.920309
4,122.50,264.263635
5,113.50,200.282417
6,217.00,222.963575
7,5.75,270.473911
8,500.96,264.035300
9,836.00,299.130046


,ActualFare,PredictedFare,Residual,AbsoluteResidual
0,300.50,308.147816,-7.647816,7.647816
1,143.50,348.144184,-204.644184,204.644184
2,67.50,115.872785,-48.372785,48.372785
3,138.00,174.920309,-36.920309,36.920309
4,122.50,264.263635,-141.763635,141.763635
5,113.50,200.282417,-86.782417,86.782417
6,217.00,222.963575,-5.963575,5.963575
7,5.75,270.473911,-264.723911,264.723911
8,500.96,264.035300,236.924700,236.924700
9,836.00,299.130046,536.869954,536.869954


even though this is only 10 rows, we can see that lower fares tend to be overpredicted and very high fares are underpredicted, with the linear regression model learning to predict closer to the centre of the target distribution and struggling with more extreme/complex values